# Can Machine Learning Estimate Beehive Weight Without a Scale?
## Empirical Investigation on 78 Honeybee Colonies in Germany (2019–2022)

**Authors:** Amha Endrayes, Natnael Behailu  
**Institution:** Robox AI and Robotics Center, Addis Ababa, Ethiopia  
**Primary Dataset:** Zenodo Record 10407693 (DOI: [10.5281/zenodo.10407693](https://doi.org/10.5281/zenodo.10407693))  
**Data Publication:** Senger et al., *Data in Brief* 52 (2024) 110015 ([PMC10809063](https://pmc.ncbi.nlm.nih.gov/articles/PMC10809063/))

---
### Project Objective
This notebook trains and evaluates machine learning models on the official BeeObserver German honeybee colony dataset to determine whether environmental and internal sensors can eliminate or reduce the need for physical hive scales.

## 1. Environment & Hardware Verification

In [ ]:
import sys, os, platform, psutil, multiprocessing
print(f"Python Version: {platform.python_version()}")
print(f"OS Platform: {platform.platform()}")
print(f"Logical CPU Cores: {multiprocessing.cpu_count()}")
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"System RAM: {ram_gb:.2f} GB")
try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    else:
        print("Running on CPU runtime (optimal for Tabular Gradient Boosted Trees).")
except ImportError:
    print("CPU runtime.")

## 2. Google Drive Mounting & Path Setup

In [ ]:
from pathlib import Path
import glob

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    base_dir = Path('/content/drive/MyDrive')
else:
    base_dir = Path('./')

# Locate real dataset directory in Drive
candidate_dirs = list(base_dir.glob('**/bob_publication_data/**/preprocessed/*_h'))
if candidate_dirs:
    data_dir = candidate_dirs[0].parent
    print(f"Found extracted dataset directory: {data_dir}")
else:
    data_dir = base_dir / 'Beehive_Weight_ML' / 'data'
    print(f"Using default project data directory: {data_dir}")

## 3. Real Colony Data Loading & Ingestion

In [ ]:
import pandas as pd, numpy as np

# Search for hourly colony CSVs
hourly_csvs = list(base_dir.glob('**/*_h/*.csv')) + list(Path('.').glob('**/*_h/*.csv')) + list(Path('downloads').glob('*.csv'))
print(f"Found {len(hourly_csvs)} hourly colony files.")

dfs = []
for f in hourly_csvs[:15]: # Load representative colonies
    try:
        d = pd.read_csv(f, low_memory=False)
        if 'time' in d.columns and 'timestamp' not in d.columns:
            d['timestamp'] = d['time']
        dfs.append(d)
    except Exception as e:
        pass

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Loaded real telemetry: {df_all.shape[0]} records across {df_all['key'].nunique()} colonies.")
else:
    print("No CSVs found locally; please verify Google Drive mount path.")

## 4. Thermodynamic & Temporal Feature Engineering

In [ ]:
df_all['dt'] = pd.to_datetime(df_all['timestamp'], errors='coerce', utc=True)
df_all = df_all.dropna(subset=['dt', 'weight_kg']).sort_values(['key', 'dt']).reset_index(drop=True)
df_all = df_all[(df_all['weight_kg'] >= 10.0) & (df_all['weight_kg'] <= 100.0)]

df_all['hour'] = df_all['dt'].dt.hour + df_all['dt'].dt.minute / 60.0
df_all['doy'] = df_all['dt'].dt.dayofyear

temp_cols = [c for c in ['t_i_1', 't_i_2', 't_i_3', 't_i_4', 't_i_5'] if c in df_all.columns]
for c in temp_cols + ['t_o', 'h', 'p', 't']:
    if c in df_all.columns:
        df_all[c] = pd.to_numeric(df_all[c], errors='coerce')

df_all['temp_internal_mean'] = df_all[temp_cols].mean(axis=1)
df_all['temp_internal_std'] = df_all[temp_cols].std(axis=1).fillna(0.0)
df_all['temp_core_periph_grad'] = df_all['t_i_3'] - 0.5 * (df_all['t_i_1'] + df_all['t_i_5'])
df_all['temp_ambient_grad'] = df_all['temp_internal_mean'] - df_all['t_o']

df_all['hour_sin'] = np.sin(2.0 * np.pi * df_all['hour'] / 24.0)
df_all['hour_cos'] = np.cos(2.0 * np.pi * df_all['hour'] / 24.0)
df_all['doy_sin'] = np.sin(2.0 * np.pi * df_all['doy'] / 365.25)
df_all['doy_cos'] = np.cos(2.0 * np.pi * df_all['doy'] / 365.25)
df_all['solar_proxy'] = np.maximum(0.0, np.cos(2 * np.pi * (df_all['hour'] - 13.0) / 24.0))

for lag in [1, 3, 6, 24]:
    df_all[f't_o_lag_{lag}'] = df_all.groupby('key')['t_o'].shift(lag)
    df_all[f't_core_lag_{lag}'] = df_all.groupby('key')['t_i_3'].shift(lag)

df_all['dt_o_1h'] = df_all['t_o'] - df_all['t_o_lag_1']

for h in [1, 3, 6, 12, 24]:
    df_all[f'delta_{h}h'] = df_all.groupby('key')['weight_kg'].diff(h)

df_clean = df_all.dropna(subset=['t_o', 'temp_internal_mean']).reset_index(drop=True)
print(f"Engineered feature dataset: {df_clean.shape[0]} rows.")

## 5. Model Evaluation: Zero-Shot vs 24-Hour Calibrated

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = ['t_i_1', 't_i_2', 't_i_3', 't_i_4', 't_i_5', 't_o', 'h', 'p',
                'temp_internal_mean', 'temp_internal_std', 'temp_core_periph_grad', 'temp_ambient_grad',
                'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'solar_proxy',
                't_o_lag_1', 't_core_lag_1', 't_o_lag_3', 't_core_lag_3', 't_o_lag_6', 't_core_lag_6', 't_o_lag_24', 't_core_lag_24', 'dt_o_1h']
feature_cols = [f for f in feature_cols if f in df_clean.columns]
df_clean[feature_cols] = df_clean[feature_cols].fillna(df_clean[feature_cols].median())

colonies = list(df_clean['key'].unique())
train_cols = colonies[:int(0.7 * len(colonies))]
test_cols = colonies[int(0.7 * len(colonies)):]

tr = df_clean[df_clean['key'].isin(train_cols)]
te = df_clean[df_clean['key'].isin(test_cols)]

model = HistGradientBoostingRegressor(max_iter=150, learning_rate=0.08, random_state=42)
model.fit(tr[feature_cols], tr['weight_kg'])

# Zero-shot
zero_preds = model.predict(te[feature_cols])
print("=== Task A: Zero-Shot on Unseen Colonies ===")
print(f"MAE: {mean_absolute_error(te['weight_kg'], zero_preds)*1000:.1f} g | R2: {r2_score(te['weight_kg'], zero_preds):.4f}")

# 24-hour calibrated
calib_records = []
for c_id in test_cols:
    sub = te[te['key'] == c_id].copy()
    if len(sub) < 30: continue
    tare = (sub.iloc[:24]['weight_kg'] - model.predict(sub.iloc[:24][feature_cols])).mean()
    sub_eval = sub.iloc[24:].copy()
    sub_eval['pred'] = model.predict(sub_eval[feature_cols]) + tare
    calib_records.append(sub_eval)

df_cal = pd.concat(calib_records)
print("=== Task A: 24h Calibrated on Unseen Colonies ===")
print(f"MAE: {mean_absolute_error(df_cal['weight_kg'], df_cal['pred'])*1000:.1f} g | R2: {r2_score(df_cal['weight_kg'], df_cal['pred']):.4f}")